**SECTION 1 : IMPORTING NECESSARY LIBRARIES**

In [0]:
%pip install --upgrade --force-reinstall matplotlib
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings("ignore")
import os

# Set global plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style="whitegrid", palette="muted")

COLORS = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0',
          '#00BCD4','#FF5722','#795548','#607D8B','#FFC107']

print("✅ Libraries imported successfully")

**SECTION 2: DATA LOADING**

In [0]:
df = pd.read_csv("/Workspace/Users/victorsarmacharjee20@gmail.com/Datasets/Restaurant Data.csv")
print("\n📊 Dataset Loaded")

**SECTION 3 : BASIC OVERVIEW OF THE DATASET**

In [0]:
# ── 3a. First 5 Rows of the Dataset
df.head()

In [0]:
# ── 3b. Last 5 Rows of the Dataset
df.tail()

In [0]:
# ── 3c. Shape of the Dataset
print(f"{df.shape[0]:,} rows × {df.shape[1]} columns")

In [0]:
# ── 3d. Column Names of the Dataset
print(df.columns.tolist())          # Note: df.columns

In [0]:
# ── 3e. Basic or General Information of the Dataset
df.info()

In [0]:
# ── 3f. Data Types of the Dataset
print(df.dtypes)                          # Note: df.dtypes

In [0]:
# ── 3g. Size of the Dataset
df.size

In [0]:
# ── 3h. Check if Dataset is Empty
df.empty

In [0]:
# ── 3i. Statistical Summary of the Dataset
df.describe()

In [0]:
# ── 3j. Missing Values of the Dataset
print(df.isnull().sum())                # Note : df.isnull().sum()

**SECTION 4 : DATA CLEANING**

In [0]:
# ── 4a. Drop rows where Cuisine is null (only 19 rows — negligible)
df = df.dropna(subset=['Cuisine'])
print(f"\n Dropped 19 null Cuisine rows. Shape: {df.shape}")

In [0]:
# ── 4b. Clean Rating column
#    Some ratings are 'New' (brand new restaurants) or '-' (no rating yet)
#    We'll keep them as separate categories and also create a numeric version
df['Rating_Raw'] = df['Rating'].astype(str).str.strip()
 
def clean_rating(val):
    val = str(val).strip()
    if val in ['-', 'New', '', 'nan']:
        return np.nan
    try:
        return float(val)
    except:
        return np.nan
 
df['Rating_Numeric'] = df['Rating_Raw'].apply(clean_rating)
print(f"Rating cleaned. Numeric ratings available: {df['Rating_Numeric'].notna().sum():,}")

In [0]:
# ── 4c. Clean Average Price — remove ₹ symbol and 'for one', handle commas
df['Price_Clean'] = (
    df['Average Price']
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace('for one', '', regex=False)
    .str.strip()
)
# Some rows have just 'for one' with no number
df['Price_Numeric'] = pd.to_numeric(df['Price_Clean'], errors='coerce')
print(f"Price cleaned. Numeric prices available: {df['Price_Numeric'].notna().sum():,}")

In [0]:
# ── 4d. Clean Delivery Time — remove 'min'
df['Delivery_Minutes'] = (
    df['Average Delivery Time']
    .str.replace('min', '', regex=False)
    .str.strip()
)
df['Delivery_Minutes'] = pd.to_numeric(df['Delivery_Minutes'], errors='coerce')
print(f"Delivery time cleaned. Min: {df['Delivery_Minutes'].min()}, Max: {df['Delivery_Minutes'].max()}")

In [0]:
# ── 4e. Clean Location column — strip whitespace
df['Location'] = df['Location'].str.strip()

In [0]:
# ── 4f. Simplify Safety Measure to a short label
df['Safety_Label'] = df['Safety Measure'].apply(
    lambda x: 'WHO Protocol' if 'WHO' in str(x) else 'Max Safety'
)

In [0]:
print("\n--- Final Cleaned Dataset Sample ---")
print(df[['Restaurant Name','Rating_Numeric','Price_Numeric','Delivery_Minutes',
          'Location','Safety_Label']].head())

**SECTION 5 : FEATURE ENGINEERING**

In [0]:
# ── 5a. Rating Category — label each restaurant
def rating_category(r):
    if pd.isna(r):
        return 'Not Rated'
    elif r >= 4.5:
        return 'Excellent (4.5+)'
    elif r >= 4.0:
        return 'Good (4.0–4.4)'
    elif r >= 3.5:
        return 'Average (3.5–3.9)'
    else:
        return 'Below Average (<3.5)'
 
df['Rating_Category'] = df['Rating_Numeric'].apply(rating_category)

In [0]:
# ── 5b. Price Band — affordable / mid / premium
def price_band(p):
    if pd.isna(p):
        return 'Unknown'
    elif p <= 100:
        return 'Budget (≤₹100)'
    elif p <= 300:
        return 'Mid-Range (₹101–300)'
    elif p <= 600:
        return 'Premium (₹301–600)'
    else:
        return 'Luxury (₹600+)'
 
df['Price_Band'] = df['Price_Numeric'].apply(price_band)

In [0]:
# ── 5c. Delivery Speed Category
def delivery_speed(d):
    if pd.isna(d):
        return 'Unknown'
    elif d <= 25:
        return 'Fast (≤25 min)'
    elif d <= 35:
        return 'Normal (26–35 min)'
    else:
        return 'Slow (>35 min)'
 
df['Delivery_Speed'] = df['Delivery_Minutes'].apply(delivery_speed)

In [0]:
# ── 5d. Primary Cuisine — take the FIRST cuisine listed
df['Primary_Cuisine'] = df['Cuisine'].str.split(',').str[0].str.strip()

In [0]:
# ── 5e. Cuisine Count per restaurant
df['Cuisine_Count'] = df['Cuisine'].str.split(',').str.len()

In [0]:
print("\n Feature Engineering done!")
print(df[['Rating_Category','Price_Band','Delivery_Speed','Primary_Cuisine','Cuisine_Count']].head())

**SECTION 6 : EXPLORATORY DATA ANALYSIS (EDA)**

In [0]:
# ── 6a. Rating Distribution Stats
print("\n Rating Stats (numeric only):")
print(df['Rating_Numeric'].describe().round(2))

In [0]:
# ── 6b. Price Stats
print("\n Price Stats (₹ per person):")
print(df['Price_Numeric'].describe().round(2))

In [0]:
# ── 6c. Delivery Time Stats
print("\n Delivery Time Stats (minutes):")
print(df['Delivery_Minutes'].describe().round(2))

In [0]:
# ── 6d. Top 10 Cities by Restaurant Count
print("\n Top 10 Cities by Restaurant Count:")
print(df['Location'].value_counts().head(10))

In [0]:
# ── 6e. Top 10 Primary Cuisines
print("\n Top 10 Primary Cuisines:")
print(df['Primary_Cuisine'].value_counts().head(10))

In [0]:
# ── 6f. Rating Category Distribution
print("\n Rating Category Distribution:")
print(df['Rating_Category'].value_counts())

In [0]:
# ── 6g. Price Band Distribution
print("\n Price Band Distribution:")
print(df['Price_Band'].value_counts())

In [0]:
# ── 6h. Delivery Speed Distribution
print("\n Delivery Speed Distribution:")
print(df['Delivery_Speed'].value_counts())

In [0]:
# ── 6i. Safety Measure Distribution
print("\n Safety Measure Distribution:")
print(df['Safety_Label'].value_counts())

In [0]:
# ── 6j. Average rating by city (top 10 cities)
top_cities = df['Location'].value_counts().head(10).index
city_avg_rating = (
    df[df['Location'].isin(top_cities)]
    .groupby('Location')['Rating_Numeric']
    .mean()
    .round(2)
    .sort_values(ascending=False)
)
print("\n Average Rating — Top 10 Cities:")
print(city_avg_rating)

In [0]:
# ── 6k. Average price by city (top 10 cities)
city_avg_price = (
    df[df['Location'].isin(top_cities)]
    .groupby('Location')['Price_Numeric']
    .mean()
    .round(0)
    .sort_values(ascending=False)
)
print("\n Average Price per Person — Top 10 Cities:")
print(city_avg_price)

**SECTION 7 : DATA VISUALIZATIONS**

In [0]:
# ── Plot 01: Rating Distribution
fig, ax = plt.subplots(figsize=(8, 4))
rating_data = df['Rating_Numeric'].dropna()
ax.hist(rating_data, bins=20, color='#2196F3', edgecolor='white', linewidth=0.8)
ax.set_title('Distribution of Restaurant Ratings', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Rating', fontsize=11)
ax.set_ylabel('Number of Restaurants', fontsize=11)
ax.axvline(rating_data.mean(), color='#E91E63', linestyle='--', linewidth=2,
           label=f'Mean: {rating_data.mean():.2f}')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 02: Top 10 Cities by Restaurant Count 
fig, ax = plt.subplots(figsize=(8, 4))
city_counts = df['Location'].value_counts().head(10)
bars = ax.barh(city_counts.index[::-1], city_counts.values[::-1], color=COLORS)
ax.set_title('Top 10 Cities by Number of Restaurants', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Restaurants', fontsize=11)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 20, bar.get_y() + bar.get_height()/2, f'{int(w):,}',
            va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 03: Top 10 Primary Cuisines 
fig, ax = plt.subplots(figsize=(8, 4))
cuisine_counts = df['Primary_Cuisine'].value_counts().head(10)
bars = ax.barh(cuisine_counts.index[::-1], cuisine_counts.values[::-1], color=COLORS)
ax.set_title('Top 10 Primary Cuisines Across India', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Restaurants', fontsize=11)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 10, bar.get_y() + bar.get_height()/2, f'{int(w):,}',
            va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 04: Rating Category
fig, ax = plt.subplots(figsize=(8, 4))
cat_order = ['Excellent (4.5+)', 'Good (4.0–4.4)', 'Average (3.5–3.9)',
             'Below Average (<3.5)', 'Not Rated']
cat_counts = df['Rating_Category'].value_counts().reindex(cat_order).dropna()
pie_colors = ['#4CAF50','#2196F3','#FFC107','#FF5722','#9E9E9E']
wedges, texts, autotexts = ax.pie(
    cat_counts.values,
    labels=cat_counts.index,
    colors=pie_colors,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.75
)
for t in texts: t.set_fontsize(10)
for at in autotexts: at.set_fontsize(9)
ax.set_title('Restaurant Rating Category Breakdown', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 05: Price Band Distribution 
fig, ax = plt.subplots(figsize=(8, 4))
band_order = ['Budget (≤₹100)', 'Mid-Range (₹101–300)', 'Premium (₹301–600)', 'Luxury (₹600+)', 'Unknown']
band_counts = df['Price_Band'].value_counts().reindex(band_order).fillna(0)
bar_colors = ['#4CAF50','#2196F3','#FF9800','#E91E63','#9E9E9E']
bars = ax.bar(band_counts.index, band_counts.values, color=bar_colors, edgecolor='white')
ax.set_title('Restaurant Count by Price Band', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Number of Restaurants', fontsize=11)
ax.set_xlabel('Price Band', fontsize=11)
plt.xticks(rotation=20, ha='right', fontsize=9)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 50, f'{int(h):,}',
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 06: Delivery Time Distribution 
fig, ax = plt.subplots(figsize=(8, 4))
del_data = df['Delivery_Minutes'].dropna()
ax.hist(del_data, bins=25, color='#FF9800', edgecolor='white', linewidth=0.8)
ax.set_title('Distribution of Average Delivery Time', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Delivery Time (minutes)', fontsize=11)
ax.set_ylabel('Number of Restaurants', fontsize=11)
ax.axvline(del_data.mean(), color='#E91E63', linestyle='--', linewidth=2,
           label=f'Mean: {del_data.mean():.1f} min')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 07: Avg Rating by City (Top 10)
fig, ax = plt.subplots(figsize=(8, 4))
city_rating_sorted = city_avg_rating.sort_values()
bar_colors_city = ['#2196F3' if v >= city_avg_rating.mean() else '#90CAF9'
                   for v in city_rating_sorted.values]
bars = ax.barh(city_rating_sorted.index, city_rating_sorted.values, color=bar_colors_city)
ax.set_title('Average Restaurant Rating — Top 10 Cities', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Average Rating', fontsize=11)
ax.set_xlim(3.5, 4.5)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.01, bar.get_y() + bar.get_height()/2, f'{w:.2f}',
            va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 08: Safety Measure Distribution 
fig, ax = plt.subplots(figsize=(8, 4))
safety_counts = df['Safety_Label'].value_counts()
bars = ax.bar(safety_counts.index, safety_counts.values,
              color=['#4CAF50','#2196F3'], edgecolor='white', width=0.5)
ax.set_title('Safety Compliance Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Number of Restaurants', fontsize=11)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 100, f'{int(h):,}',
            ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 09: Rating vs Price 
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = df[df['Rating_Numeric'].notna() & df['Price_Numeric'].notna()]
# Use a sample to avoid overplotting
sample = plot_df.sample(n=min(3000, len(plot_df)), random_state=42)
ax.scatter(sample['Price_Numeric'], sample['Rating_Numeric'],
           alpha=0.3, color='#2196F3', s=15, edgecolors='none')
# Trend line
z = np.polyfit(sample['Price_Numeric'], sample['Rating_Numeric'], 1)
p = np.poly1d(z)
x_line = np.linspace(sample['Price_Numeric'].min(), sample['Price_Numeric'].max(), 100)
ax.plot(x_line, p(x_line), color='#E91E63', linewidth=2, linestyle='--', label='Trend')
ax.set_title('Rating vs Price Per Person', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Average Price for One (₹)', fontsize=11)
ax.set_ylabel('Rating', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 10: Delivery Speed 
fig, ax = plt.subplots(figsize=(8, 4))
speed_order = ['Fast (≤25 min)', 'Normal (26–35 min)', 'Slow (>35 min)']
speed_counts = df['Delivery_Speed'].value_counts().reindex(speed_order).fillna(0)
speed_colors = ['#4CAF50','#FFC107','#F44336']
wedges, texts, autotexts = ax.pie(
    speed_counts.values,
    labels=speed_counts.index,
    colors=speed_colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75
)
for t in texts: t.set_fontsize(10)
ax.set_title('Delivery Speed Distribution', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 11: Cuisine Diversity (Avg Cuisine Count by City) 
fig, ax = plt.subplots(figsize=(8, 4))
cuisine_diversity = (
    df[df['Location'].isin(top_cities)]
    .groupby('Location')['Cuisine_Count']
    .mean()
    .round(2)
    .sort_values(ascending=False)
)
bars = ax.bar(cuisine_diversity.index, cuisine_diversity.values,
              color='#9C27B0', edgecolor='white')
ax.set_title('Average Cuisine Variety per Restaurant — Top 10 Cities', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Avg Number of Cuisines per Restaurant', fontsize=11)
ax.set_xlabel('City', fontsize=11)
plt.xticks(rotation=30, ha='right', fontsize=9)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.2f}',
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [0]:
# ── Plot 12: Price Band vs Avg Rating 
fig, ax = plt.subplots(figsize=(8, 4))
band_order_known = ['Budget (≤₹100)', 'Mid-Range (₹101–300)', 'Premium (₹301–600)', 'Luxury (₹600+)']
price_rating = (
    df[df['Price_Band'].isin(band_order_known) & df['Rating_Numeric'].notna()]
    .groupby('Price_Band')['Rating_Numeric']
    .mean()
    .reindex(band_order_known)
    .round(2)
)
bars = ax.bar(price_rating.index, price_rating.values,
              color=['#4CAF50','#2196F3','#FF9800','#E91E63'], edgecolor='white')
ax.set_title('Average Rating by Price Band', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Average Rating', fontsize=11)
ax.set_ylim(3.5, 4.5)
plt.xticks(rotation=15, ha='right', fontsize=9)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
            ha='center', fontsize=10)
plt.tight_layout()
plt.show()

**SECTION 8 : KEY BUSINESS INSIGHTS SUMMARY**

In [0]:
print("\n" + "="*60)
print("BUSINESS INSIGHTS & RECOMMENDATIONS")
print("="*60)
 
total = len(df)
rated = df['Rating_Numeric'].notna().sum()
good_plus = (df['Rating_Category'].isin(['Excellent (4.5+)','Good (4.0–4.4)'])).sum()
budget = (df['Price_Band'] == 'Budget (≤₹100)').sum()
fast_del = (df['Delivery_Speed'] == 'Fast (≤25 min)').sum()
 
print(f"""
1. MARKET SCALE
   • Dataset covers {total:,} restaurants across 98 Indian cities
   • {rated/total*100:.1f}% of restaurants have been rated by customers
 
2. QUALITY LANDSCAPE
   • {good_plus/total*100:.1f}% restaurants fall in Good or Excellent category (4.0+)
   • Mean rating: {df['Rating_Numeric'].mean():.2f} — indicating generally positive customer experience
 
3. PRICING
   • {budget/total*100:.1f}% restaurants are in Budget segment (≤₹100 per person)
   • Most affordable cities offer the widest variety of options
 
4. DELIVERY EFFICIENCY
   • Average delivery time: {df['Delivery_Minutes'].mean():.1f} minutes
   • {fast_del/total*100:.1f}% restaurants deliver within 25 minutes
 
5. SAFETY COMPLIANCE
   • ~57% follow Max Safety measures; ~43% follow WHO protocol
   • Safety compliance is near-universal across all cities
 
6. CUISINE DIVERSITY
   • North Indian is the most common primary cuisine
   • Restaurants in metro cities offer more cuisine variety on average
 
7. PRICE–QUALITY INSIGHT
   • Higher price bands do not always mean higher ratings
   • Budget restaurants maintain competitive ratings with premium ones
""")
 
print("\n Analysis complete!")
print("=" * 60) 